# Baseline Setup and Smoke Test
Can one normal bottle reference support end-to-end scoring of the 83 bottle test images? We reconstruct the result of the original GPU smoke test from its archived predictions. The original smoke test used the dataset revision in the run record; the later baseline used the verified official archive.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
import hashlib
from sklearn.metrics import roc_auc_score
folder = root / "results/bottle_smoke_test/bottle_1shot_seed0"
record = json.loads((folder / "metrics.json").read_text())
display(pd.Series({k: record[k] for k in ["model", "resolution", "shots", "seed", "reference_image", "rotation", "masking", "dataset_revision", "checkpoint_sha256"]}).to_frame("Run settings"))

,Run settings
model,dinov2_vits14
resolution,448
shots,1
seed,0
reference_image,000.png
rotation,True
masking,False
dataset_revision,c75b39616f84db43677bcc8228caaafaf5096d7f
checkpoint_sha256,b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d...


## From reference to image score
The frozen model represents the reference as patch features. Rotation augmentation expands the reference bank. Each query patch is compared with the nearest normal feature. The highest-distance patches contribute to the image score. The CSV stores the resulting scores from inference; this cell checks their provenance and ranking.

In [2]:
predictions = folder / "predictions.csv"
assert hashlib.sha256(predictions.read_bytes()).hexdigest() == "45d8030895b1ba7c25a025c8fa5551749ea61f5f564903db40d2738325bdaccd"
scores = pd.read_csv(predictions)
assert scores.image.is_unique and len(scores) == record["test_images"]
assert scores.label.value_counts().to_dict() == {0: 20, 1: 63}
assert np.array_equal(scores.label, (~scores.image.str.startswith("good/")).astype(int))
auc = roc_auc_score(scores.label, scores.anomaly_score)
assert abs(auc - record["image_auroc"]) < 1e-12
display(scores.groupby("label").anomaly_score.agg(["count", "min", "median", "max"]).rename(index={0:"Normal", 1:"Defective"}))
print(f"Recomputed image AUROC: {auc:.12f}")

,count,min,median,max
label,,,,
Normal,20,0.135813,0.159494,0.244535
Defective,63,0.224727,0.385942,0.733876


Recomputed image AUROC: 0.998412698413


## Which pairs overlap?
At AUROC close to one, inspecting the actual pairs is more informative than rounding the score to 100%.

In [3]:
normal = scores.query("label == 0")
defective = scores.query("label == 1")
pairs = defective.merge(normal, how="cross", suffixes=("_defect", "_normal"))
overlap = pairs.query("anomaly_score_defect <= anomaly_score_normal")
display(overlap[["image_defect", "anomaly_score_defect", "image_normal", "anomaly_score_normal"]].reset_index(drop=True))

,image_defect,anomaly_score_defect,image_normal,anomaly_score_normal
0,broken_small/000.png,0.228442,good/001.png,0.244535
1,contamination/019.png,0.224727,good/001.png,0.244535


This test verifies image scoring for one reference in one category. Localization and cross-category reproducibility are addressed in [Baseline Reproduction](baseline_reproduction.ipynb). [Original smoke-test record](../research_logs/bottle_smoke_test.md).